# TL-mesh lab

Triangular L-meshes (TL-meshes) from the triangle meshes in `data/THlmeshes_input/TL/`:
oriented spheres on `top.vertices` (`cv`, `rv`), oriented planes on `top.faces` (`npf`, `hpf`).
`tl_mesh_computation(name, tau)` initialises the L-mesh (sphere radius τ × smallest
curvature-sphere radius; boundary spheres take the mean radius of their interior
neighbours), optimises it, runs `improve_contact()` and exports to
`notebooks/out/lmesh/TL_results/<mesh>/exp_<τ>/`. Choose the mesh and τ in the last line of the
setup cell; the batch sweep runs every mesh at several τ.

Fairness terms used here:

| type | strips | faces |
|------|--------|-------|
| tri  | `TLmesh_reg_strips` | `TLmesh_reg_hex_faces` |

Quad L-meshes get their own notebook; hexagonal L-meshes are handled by `hl_mesh_lab.ipynb`.

**Viewer:** `kv.init()` starts a kayviz viewer and prints its URL; open it in a browser. To share one viewer between several notebooks, run `python -m kayviz` in a terminal first.

In [3]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for sub in ("",):  # project root: LMesh/, Lconjugacy/ (hanan, kayviz are installed)
    p = os.path.join(PROJECT_ROOT, sub) if sub else PROJECT_ROOT
    if p not in sys.path:
        sys.path.insert(0, p)

print("Project root:", PROJECT_ROOT)

from LMesh.lmesh_serializers import lmesh_visualize
import kayviz as kv
import pymeshlab
from LMesh.L_mesh import Lmesh
from hanan.geometry import Mesh
from hanan.optimization.optimizer import Optimizer
from hanan.geometry.utils import (
    read_obj,
    read_off,
    normalize_vertices,
    compute_cone_axes,
    conical_mesh_smallest_curvature_sphere
)
import numpy as np
import igl

from hanan.optimization.optimizer import Optimizer
from LMesh.Optimization.Or_contactFree import Or_contactFree
from LMesh.Optimization.TLmesh_reg_strips import TLmesh_reg_strips
from LMesh.Optimization.TLmesh_reg_hex_faces import TLmesh_reg_hex_faces
from LMesh.Optimization.Tangential_Pos import TDPositive
from LMesh.Optimization.Tangential_Dist import Tangential_Dist
from LMesh.Optimization.ContactPointsProximity import ContactPointsProximity
from LMesh.Optimization.ContactPointsGlide import ContactPointsGlide

# Tri inputs are the *.out triangular meshes; their reference surfaces (same
# basename without ".out") live next to the PH inputs.
TRI_DATA = os.path.join(PROJECT_ROOT, "data", "THlmeshes_input", "TL")
REF_DATA = os.path.join(PROJECT_ROOT, "data", "THlmeshes_input", "PH")
OUT_PATH = os.path.join(PROJECT_ROOT,  "notebooks", "out", "lmesh", "TL_results")



def interior_mean_radii(mesh, radii):
    """Boundary radii <- mean radius of the interior 1-ring neighbours.

    Vertices with no interior neighbour (e.g. corners) take the mean of their
    already-set boundary neighbours, repeated until every one is set.
    """
    radii = radii.copy()
    adj = mesh.vertex_adjacency_list()
    bd = set(mesh.boundary_vertices().tolist())
    todo = []
    for v in bd:
        inner = [u for u in adj[v] if u not in bd]
        if inner:
            radii[v] = radii[inner].mean()
        else:
            todo.append(v)
    done = bd - set(todo)
    while todo:
        left = []
        for v in todo:
            nb = [u for u in adj[v] if u in done]
            if nb:
                radii[v] = radii[nb].mean()
            else:
                left.append(v)
        if len(left) == len(todo):
            break  # isolated boundary component: keep the curvature radii
        done |= set(todo) - set(left)
        todo = left
    return radii


def setup_optimizer(lmesh, Vr=None, Fr=None):
    """Build the optimizer with the triangular-L-mesh fairness terms.

    Swapped convention (branch swapSP): oriented spheres live on top.vertices
    (variables cv, rv) and oriented planes on top.faces (variables npf, hpf).
    """
    opt = Optimizer()

    weights = {
        "Or_Contact": 1.0,
        "TD": 0,
        "TDpos": 0,
        "Reg_Strips": 5e-3,
        "Reg_Faces": 5e-3,
        "contactProximity": 1e-4,
        "contactGliding": 1e-4,
        "npf_fairness": 1e-3
    }

    cf  = lmesh.top.vertices[:, :3]
    rf  = lmesh.top.vertices[:, 3]
    npf = lmesh.PF[:, :3]
    hpf = lmesh.PF[:, 3]

    # npf/hpf are per-FACE variables (one oriented plane per L-mesh face),
    # so fairness must use face->face adjacency.
    plane_adj = lmesh.top.face_face_adjacency_list()
    si_sj_e   = lmesh.top.edge_vertices()

    opt.add_variable("cv", cf.flatten())
    opt.add_variable("rv", rf)
    opt.add_variable("npf", npf.flatten())
    opt.add_variable("hpf", hpf.flatten())

    # Oriented contact term
    Or_contact = Or_contactFree()
    Or_contact.name = "or_contact"
    opt.add_objective_term(
        Or_contact,
        args=(lmesh.top,),
        w=weights["Or_Contact"],
        ce=1,
    )

    # Strips: runs of three consecutive spheres in the 6-ring of a sphere.
    reg_strips = TLmesh_reg_strips()
    reg_strips.name = "triRegStrips"
    opt.add_objective_term(
        reg_strips,
        args=(lmesh.top,),
        w=weights["Reg_Strips"],
        ce=1,
    )

    # Faces: rings of face-planes around two adjacent spheres.
    reg_faces = TLmesh_reg_hex_faces()
    reg_faces.name = "triRegFaces"
    opt.add_objective_term(
        reg_faces,
        args=(lmesh.top,),
        w=weights["Reg_Faces"],
        ce=1,
    )

    Tangential_Dist_term = Tangential_Dist()
    Tangential_Dist_term.name = "tangential_dist"
    opt.add_objective_term(
        Tangential_Dist_term,
        args=(si_sj_e, ),
        w=weights["TD"],
        ce=1,
    )

    Tangential_Dist_pos_term = Tangential_Dist()
    Tangential_Dist_pos_term.name = "tangential_dist_pos"
    opt.add_objective_term(
        Tangential_Dist_pos_term,
        args=(si_sj_e, ),
        w=weights["TDpos"],
        ce=1,
    )

    
    ContactPointsProximity_term = ContactPointsProximity()
    ContactPointsProximity_term.name = "contactProximity"
    opt.add_objective_term(
        ContactPointsProximity_term,
        args=(lmesh.top.vertex_face_adjacency_list(), Vr, Fr),
        w=weights["contactProximity"],
        ce=1,
    )

    ContactPointsGlide_term = ContactPointsGlide()
    ContactPointsGlide_term.name = "contactGliding"
    opt.add_objective_term(
        ContactPointsGlide_term,
        args=(lmesh.top.vertex_face_adjacency_list(), Vr, Fr),
        w=weights["contactGliding"],
        ce=1,
    )

    # Normal vector fairness
    opt.set_fairness("npf", plane_adj, 3, 0.1, 10, weights["npf_fairness"])
    opt.unitize_variable("npf", 3, 10)
    opt.initialize_optimizer(verbose=True, adaptive_mu=False)

    return opt


def ref_surface_mesh(V, F):
    """Remesh + smooth a reference surface for the proximity/glide terms."""
    m  = pymeshlab.Mesh(vertex_matrix=V, face_matrix=F)
    ms = pymeshlab.MeshSet()
    ms.add_mesh(m)

    ms.apply_filter("meshing_isotropic_explicit_remeshing",
                    targetlen=pymeshlab.PercentageValue(1.0),  # 1% of bbox diagonal
                    iterations=5)

    ms.apply_filter("apply_coord_taubin_smoothing", lambda_=0.5, mu=-0.53, stepsmoothnum=10)

    mesh = ms.current_mesh()
    return mesh.vertex_matrix(), mesh.face_matrix()


# These meshes are stored with outward-pointing normals while the rest point
# inward; reversing the face winding flips them so every input agrees.  The
# sphere centres are offset along the vertex normals, so a mesh loaded with the
# wrong winding gets its spheres placed on the wrong side of the surface.
# (Checked via signed volume: positive => outward under this winding.)
def _signed_volume(V, faces):
    vol = 0.0
    for f in faces:
        f = list(f)
        for k in range(1, len(f) - 1):
            vol += np.dot(V[f[0]], np.cross(V[f[k]], V[f[k + 1]])) / 6.0
    return vol


def load_oriented(name, data_dir):
    """Read a mesh and return (V, F) with consistently inward-pointing normals.

    The vertices are not renormalised, so they stay in the frame of the
    reference surface.  The few non-triangular faces at the singularities of
    the PH-CPF meshes are fan-triangulated.
    """
    V, F = read_obj(os.path.join(data_dir, name + ".obj"))
    #V = normalize_vertices(V, 2)
    F = [[f[0], f[k], f[k + 1]] for f in F for k in range(1, len(f) - 1)]
    if _signed_volume(V, F) > 0:
        F = [f[::-1] for f in F]     # reverse winding -> flip normals
        print(f"    [orient] flipped normals for {name}")
    return V, np.asarray(F, dtype=np.int64)


_REF_CACHE = {}

def ref_pair(name, data_dir=TRI_DATA):
    """Reference surface for the proximity/glide terms (not renormalised).

    The "*.out" meshes use the same basename without ".out", looked up in
    data_dir and then REF_DATA; a mesh without one is its own reference.
    Cached: it is the same surface for every tau of a given mesh.
    """
    if name in _REF_CACHE:
        return _REF_CACHE[name]
    ref_name = name[:-4] if name.endswith(".out") else name
    for d in (data_dir, REF_DATA):
        path = os.path.join(d, ref_name + ".obj")
        if os.path.exists(path):
            break
    Vr, Fr = read_obj(path)
    Fr = np.array([[f[0], f[k], f[k + 1]] for f in Fr for k in range(1, len(f) - 1)])
    #Vr, Fr = ref_surface_mesh(Vr, Fr)
    #Vr = normalize_vertices(Vr, 2)
    _REF_CACHE[name] = (np.asarray(Vr, dtype=np.float64), Fr)
    return _REF_CACHE[name]


def write_reference(exp_dir, Vr, Fr):
    with open(os.path.join(exp_dir, "reference_surface.obj"), "w") as fh:
        fh.writelines(f"v {x:.10g} {y:.10g} {z:.10g}\n" for x, y, z in Vr)
        fh.writelines(f"f {i+1} {j+1} {k+1}\n" for i, j, k in Fr)


def tl_mesh_computation(name, tau, n_iter=10):
    """Build a tri L-mesh from `name` and run the swapped-convention optimisation.

    name   : mesh basename (without .obj) inside TRI_DATA
    tau    : fraction of the smallest curvature sphere radius; boundary spheres
             take the mean radius of their interior neighbours
    n_iter : optimisation steps

    Exports to OUT_PATH/[name]/exp_[tau]/, plus the reference surface used by
    the proximity/glide terms.
    """
    exp_dir = os.path.join(OUT_PATH, name, f"exp_{tau:.2f}")
    os.makedirs(exp_dir, exist_ok=True)

    print("Reading mesh:", name)
    V, F = load_oriented(name, TRI_DATA)
    Vr, Fr = ref_pair(name, TRI_DATA)

    kv.register_surface_mesh("Input Mesh", V, F, color=(0.1, 0.1, 0.1),
                             show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)

    kv.register_surface_mesh("Reference Mesh", Vr, Fr, color=(0.8, 0.8, 0.8),
                             show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)

    basemesh = Mesh()
    basemesh.make_mesh(V, F)

    _, _, k1, k2, _ = igl.principal_curvature(V, F)
    min_curvature = np.maximum( np.abs(k1), np.abs(k2))
    smallest_radii = 1.0 / np.abs(min_curvature)
    radii = interior_mean_radii(basemesh, tau * smallest_radii)

    normals = basemesh.vertex_normals
    centers = basemesh.vertices + radii[:, np.newaxis] * normals

    lmesh = Lmesh()
    lmesh.init_L_mesh(
        basemesh,
        orientation=1,
        combinatorics="primal",
        mode="SphCongruence",
        radii=radii,
        centers=centers,
    )

    lmesh_visualize(lmesh)
    lmesh.export_data(exp_dir, "init")
    write_reference(exp_dir, Vr, Fr)

    opt = setup_optimizer(lmesh, Vr, Fr)

    for _ in range(n_iter):
        opt.get_gradients()
        opt.optimize_step()

    # Unpack optimized values
    rf_opt  = opt.unpack("rv")
    npf_opt = opt.unpack("npf").reshape(-1, 3)
    hpf_opt = opt.unpack("hpf")
    cf_opt  = opt.unpack("cv").reshape(-1, 3)

    # Update L-mesh with optimized centers (cf), radii (rf), and planes (npf, hpf)
    lmesh.update_spheres_planes(cf_opt, rf_opt, npf_opt, hpf_opt)

    opt.print_report()
    lmesh.improve_contact()
    print(f"Oriented contact deviation: {lmesh.compute_oriented_contact_dev():.3e}")
    lmesh.export_data(exp_dir, "lmesh")
    print("Exported to", exp_dir)

    lmesh_visualize(lmesh)

    return lmesh, opt


kv.init()   # pings the server; raises if it is not running

# --- TRI inputs: the *.out triangular meshes in data/THlmeshes_input/TL ---
tri_mesh_names = [
    "bumpy_patch.out",
    "coiled_torus_segment.out",
    "flared_nautilus.out",
    "helical_superellipsoid_tube.out",
    "loft_00014.out",
    "nautilus.out",
    "ribbed_conch.out",
    "seashell.out",
    "shell_00001.out",
    "shell_00002.out",
    "shell_nautilus.out",
    "testSimpleOptimized.out",
    "triMeshTest",
    "tube.out",
    "turritella_shell.out"
]

kv.clear()
#tl_mesh_computation(tri_mesh_names[1], 0.8)
#kv.show()

Project root: /home/anthony/KAUSTLocal/Laguerre_project


## Batch sweep — energies, contact deviation, LaTeX tables, view grids

Runs every mesh in `tri_mesh_names` at `tau in [0.5, 0.7, 0.85]`.

Outputs, one set per tau:

| what | where |
|------|-------|
| L-mesh exports | `notebooks/out/TL_results/[mesh]/exp_[tau]/` |
| LaTeX table | `out/tables/tl_energy_tau[TAU].tex` |
| images | `out/images/tl_images/` |

Images are split by producing notebook: `tl_images/` here, `ph_images/` for the
HL sweep in `hl_mesh_lab.ipynb`. Tables use the matching `tl_` / `ph_` prefix.

Needs an open viewer tab for the screenshots (`kv.show()`).

In [4]:
from LMesh.lmesh_serializers import lmesh_visualize
# ── Batch sweep: every tri mesh x every tau, with LaTeX tables and view grids ──
#
# For each (mesh, tau):
#   * build the tri L-mesh, run N_STEPS optimisation steps
#   * record the final optimisation energy (total and per term)
#   * run improve_contact() and record the oriented contact deviation
#   * export the initial and optimised states
#   * capture two views of the scene
#
# Outputs, one set per tau value:
#   out/tables/tl_energy_tau[TAU].tex        -- LaTeX table, ready to \input
#   out/images/tl_images/grid_tau[TAU].png   -- one row per mesh, two views
#   out/images/tl_images/[mesh]_[tau]_v*.png -- the individual views
#
# Images are classified by producing notebook, matching the table prefixes:
#   out/images/tl_images/  <- this notebook   (tables: tl_energy_tau*.tex)
#   out/images/ph_images/  <- hl_mesh_lab     (tables: ph_energy_tau*.tex)

import os, json, io, contextlib
import numpy as np

TAUS    = [0.5, 0.7, 0.85]
N_STEPS = 10

# ── Framing for the grid images ───────────────────────────────────────────────
# Two fixed three-quarter angles, each mesh fitted to the frame, so every row is
# sized consistently and a re-run reproduces the same pictures.
#
# ZOOM scales the auto-fit distance -- LOWER means CLOSER, less background:
#     1.05  default fit, generous margin
#     0.85  moderately tight
#     0.75  tight
#     0.65  very tight; a long/curved mesh may clip   <- current
ZOOM  = 0.65
VIEWS = [dict(v, zoom=ZOOM) for v in kv.TIGHT_VIEWS]

IMG_ROOT   = os.path.join(PROJECT_ROOT, "notebooks", "out", "lmesh", "images")
IMG_DIR    = os.path.join(IMG_ROOT, "tl_images")     # this notebook's images
PH_IMG_DIR = os.path.join(IMG_ROOT, "ph_images")     # hl_mesh_lab's images
TAB_DIR    = os.path.join(PROJECT_ROOT, "notebooks", "out", "lmesh", "tables")
# L-mesh exports land in notebooks/out/TL_results/[name]/exp_[tau]/, with the
# initial and optimised states side by side under that folder.
EXPORT_DIR = os.path.join(PROJECT_ROOT, "notebooks", "out", "lmesh", "TL_results")
for d in (IMG_DIR, PH_IMG_DIR, TAB_DIR, EXPORT_DIR):
    os.makedirs(d, exist_ok=True)


# _signed_volume, load_oriented, ref_pair, write_reference and
# interior_mean_radii are defined in the setup cell.
def run_case(name, tau, data_dir=TRI_DATA):
    """Build + optimise one (mesh, tau) tri case.  Returns a record dict."""
    V, F = load_oriented(name, data_dir)

    basemesh = Mesh()
    basemesh.make_mesh(V, F)

    # Tri meshes are seeded from principal curvature (the cone-axis route used
    # by the HL sweep needs a conical mesh, which these are not).
    _, _, k1, k2, _ = igl.principal_curvature(V, F)
    min_curvature  = np.maximum(np.abs(k1), np.abs(k2))
    smallest_radii = 1.0 / np.abs(min_curvature)
    # Boundary spheres start at the mean radius of their interior neighbours.
    radii   = interior_mean_radii(basemesh, tau * smallest_radii)
    centers = basemesh.vertices + radii[:, None] * basemesh.vertex_normals

    lmesh = Lmesh()
    lmesh.init_L_mesh(
        basemesh, orientation=1, combinatorics="primal",
        mode="SphCongruence", radii=radii, centers=centers,
    )

    # notebooks/out/TL_results/[name]/exp_[tau]/
    exp_dir = os.path.join(EXPORT_DIR, name, f"exp_{tau:.2f}")
    os.makedirs(exp_dir, exist_ok=True)

    # init_L_mesh() already ran compute_lmesh_structure(), so the initial state
    # is complete and exportable before any optimisation touches it.
    with contextlib.redirect_stdout(io.StringIO()):
        lmesh.export_data(exp_dir, "init")

    Vr, Fr = ref_pair(name, data_dir)
    write_reference(exp_dir, Vr, Fr)

    opt = setup_optimizer(lmesh, Vr, Fr)
    # Weights as registered, before the run (some terms damp theirs while optimising).
    weights = {k: float(w) for k, w in opt.get_term_weights_dict().items() if w != 0}

    for _ in range(N_STEPS):
        opt.get_gradients()
        opt.optimize_step()

    # Final energies (weighted, as the optimiser sees them)
    X = opt.X
    per_term = {n: float(t.compute_energy(X)) for n, t in opt.objective_terms.items()}
    total_e  = float(sum(per_term.values()))

    lmesh.update_spheres_planes(
        opt.unpack("cv").reshape(-1, 3), opt.unpack("rv"),
        opt.unpack("npf").reshape(-1, 3), opt.unpack("hpf"),
    )

    # Oriented contact deviation of the optimised L-mesh.
    lmesh.improve_contact()
    dev = lmesh.compute_oriented_contact_dev()

    # improve_contact() ends with compute_lmesh_structure(), so the optimised
    # state is fully rebuilt here and safe to export.
    with contextlib.redirect_stdout(io.StringIO()):
        lmesh.export_data(exp_dir, "lmesh")
    print(f"    exported -> {os.path.relpath(exp_dir, PROJECT_ROOT)}")

    return {
        "mesh": name, "tau": tau, "export_dir": exp_dir,
        "V": int(lmesh.top.V), "F": int(lmesh.top.F),
        "energy_total": total_e, "energy_terms": per_term,
        "contact_dev": dev,
        "n_contact_points": len(lmesh.get_contact_points()),
        "weights": weights, "n_steps": N_STEPS,
        "lmesh": lmesh,
    }


def latex_table(records, tau):
    """Render one tau's records as a LaTeX table (booktabs)."""
    def esc(s):
        return str(s).replace("_", r"\_")

    lines = []
    lines.append("% Generated by tl_mesh.ipynb -- tau = " + ("%.2f" % tau))
    lines.append(r"% Requires: \usepackage{booktabs}")
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"  \centering")
    lines.append(r"  \small")
    lines.append(r"  \begin{tabular}{lrrrr}")
    lines.append(r"    \toprule")
    head = ["Mesh", r"$|V|$", r"$|F|$", r"$E_{\mathrm{total}}$",
            r"$\delta_{\mathrm{contact}}$"]
    lines.append("    " + " & ".join(head) + r" \\")
    lines.append(r"    \midrule")
    for r in records:
        row = [esc(r["mesh"]), str(r["V"]), str(r["F"]),
               "%.3e" % r["energy_total"], "%.3e" % r["contact_dev"]]
        lines.append("    " + " & ".join(row) + r" \\")
    lines.append(r"    \bottomrule")
    lines.append(r"  \end{tabular}")
    # Steps and weights as recorded by run_case; every mesh shares one setup,
    # so a single list is printed unless they differ.
    n_steps = sorted({r["n_steps"] for r in records})
    w_sets  = [r["weights"] for r in records]
    def fmt_w(w):
        return ", ".join("\\texttt{" + esc(k) + "} $= " + ("%g" % v) + "$"
                         for k, v in w.items())
    if all(w == w_sets[0] for w in w_sets):
        w_txt = "Weights: " + fmt_w(w_sets[0]) + "."
    else:
        w_txt = "Weights: " + "; ".join(esc(r["mesh"]) + ": " + fmt_w(r["weights"])
                                         for r in records) + "."
    lines.append("  \\caption{TL-mesh optimisation after " +
                 "/".join(str(n) for n in n_steps) +
                 " steps, $\\tau = " + ("%.2f" % tau) + "$. " + w_txt + " "
                 "$\\delta_{\\mathrm{contact}}$ is the oriented contact "
                 "deviation after \\texttt{improve\\_contact()}.}")
    lines.append("  \\label{tab:tlmesh-tau-" + str(tau).replace(".", "p") + "}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


# ── Sweep ─────────────────────────────────────────────────────────────────────
all_records = {}

for tau in TAUS:
    print(f"\n{'='*70}\n  tau = {tau}\n{'='*70}")
    records, grid_rows, grid_labels = [], [], []

    for name in tri_mesh_names:
        print(f"  [{name}]  tau={tau}")
        try:
            rec = run_case(name, tau)
        except Exception as e:
            print(f"    SKIP {type(e).__name__}: {e}")
            continue

        # Draw this case alone, then capture the two views.
        kv.clear()
        lmesh_visualize(rec["lmesh"])
        try:
            blobs = kv.screenshot_views(views=VIEWS, width=640, height=480)
            for i, b in enumerate(blobs):
                fn = os.path.join(IMG_DIR, f"{name}_{tau:.2f}_v{i}.png")
                with open(fn, "wb") as fh:
                    fh.write(b)
            grid_rows.append(blobs)
            grid_labels.append(f"{name}   tau={tau:.2f}   "
                               f"E={rec['energy_total']:.2e}   "
                               f"dev={rec['contact_dev']:.2e}")
        except Exception as e:
            print(f"    (no screenshot: {e})")

        rec.pop("lmesh", None)          # don't keep meshes alive in the record
        records.append(rec)
        print(f"    E_total={rec['energy_total']:.4e}   "
              f"contact_dev={rec['contact_dev']:.4e}   "
              f"contact_pts={rec['n_contact_points']}")

    all_records[tau] = records

    # LaTeX table for this tau
    if records:
        tex  = latex_table(records, tau)
        path = os.path.join(TAB_DIR, f"tl_energy_tau{tau:.2f}.tex")
        with open(path, "w") as fh:
            fh.write(tex + "\n")
        print(f"\n  -> wrote {path}")
        print(tex)

    # Contact-sheet grid for this tau
    if grid_rows:
        gpath = os.path.join(IMG_DIR, f"grid_tau{tau:.2f}.png")
        kv.save_grid(gpath, grid_rows, labels=grid_labels)
        print(f"  -> wrote {gpath}")

# Raw numbers, in case you want to re-render the tables differently
with open(os.path.join(TAB_DIR, "tl_sweep_results.json"), "w") as fh:
    json.dump(all_records, fh, indent=2)
print(f"\nDone. Tables in {TAB_DIR}, images in {IMG_DIR}")


  tau = 0.5
  [bumpy_patch.out]  tau=0.5
    [orient] flipped normals for bumpy_patch.out
Mesh |V|=375, |F|=679, |E|=1053
Mesh |V|=375, |F|=679, |E|=1053
Adding Laplacian fairness for 679 
Thread pool: 7 workers (terms=7, cpus=32)
Levenberg-Marquardt Optimization (Simple)
Iter   | Energy        | Norm Energy   | ||dx||        | μ             | ρ        | Time    
------------------------------------------------------------------------------------------
1      | 8.747139e-04  | 1.266343e-07  | 3.912543e-01  | 4.018268e-05  | N/A      | 0.5328  
10     | 3.821352e-08  | 1.250223e-11  | 7.516164e-03  | 4.018268e-05  | N/A      | 0.1103  
L-mesh Optimized
Oriented Contact Deviation: 3.057e-11
    exported -> notebooks/out/lmesh/TL_results/bumpy_patch.out/exp_0.50
    (no screenshot: screenshot failed: no browser connected. Open the viewer tab (ws.show()) so a browser is connected.)
    E_total=3.2464e-08   contact_dev=1.7922e-29   contact_pts=2037
  [coiled_torus_segment.out]  tau=0.5
   

## Render images from saved results

Re-captures the `tl_images/` screenshots from the `lmesh.pkl` files the
sweep already wrote, so **no re-optimisation is needed**. Use this when the sweep
ran without a viewer tab connected (tables and exports land fine, images do not).

**The viewer tab must be open and connected before running this cell** — run
`kv.show()` and leave the tab open. The cell checks the connection up front and stops with a clear message
rather than silently producing nothing.

In [ ]:
from LMesh.lmesh_serializers import lmesh_visualize
# ── Render tl_images from the saved sweep results ─────────────────────────────
#
# The sweep writes exports + tables even with no browser attached, but the
# screenshots need a live viewer tab.  This cell rebuilds each optimised L-mesh
# from notebooks/out/TL_results/[mesh]/exp_[tau]/lmesh.pkl and captures
# the same two views, so images can be produced without re-running the
# optimisation.
#
# Requires an open, connected viewer tab:  kv.show()

import os, json, io, contextlib
import requests
from LMesh.L_mesh import load_lmesh

kv.show()

# ── Fail early and clearly if no browser is attached ──────────────────────────
# kv.screenshot_* raises per call; without this check the loop would print one
# "(no screenshot)" line per case and quietly leave tl_images/ empty.
_probe = requests.post(f"{kv.url()}/api/script/screenshot/request",
                       json={"views": [{"dir": [1, 0.7, 1], "up": [0, 1, 0],
                                        "zoom": 1.0}],
                             "width": 32, "height": 32},
                       timeout=10).json()
if not _probe.get("ok"):
    raise RuntimeError(
        f"No browser connected ({_probe.get('error')}). "
        "Run kv.show() and leave the viewer tab open, then re-run this cell."
    )
print("viewer connected — capturing")

# Reuse TAUS / VIEWS / IMG_DIR / EXPORT_DIR from the sweep cell above.
sweep_json = os.path.join(TAB_DIR, "tl_sweep_results.json")
recs_by_tau = {}
if os.path.exists(sweep_json):
    with open(sweep_json) as fh:
        recs_by_tau = json.load(fh)

for tau in TAUS:
    print(f"\n{'='*70}\n  tau = {tau}\n{'='*70}")
    grid_rows, grid_labels = [], []

    # Label rows with the energies the sweep recorded, when available.
    by_name = {r["mesh"]: r for r in recs_by_tau.get(str(tau), [])}

    for name in tri_mesh_names:
        pkl = os.path.join(EXPORT_DIR, name, f"exp_{tau:.2f}",
                           "lmesh.pkl")
        if not os.path.exists(pkl):
            print(f"  [{name}] no saved result for tau={tau:.2f} — skipped")
            continue

        # load_lmesh() recomputes the full structure (contact points, panels).
        with contextlib.redirect_stdout(io.StringIO()):
            lmesh = load_lmesh(pkl)
            kv.clear()
            lmesh_visualize(lmesh)

        try:
            blobs = kv.screenshot_views(views=VIEWS, width=640, height=480)
        except Exception as e:
            print(f"  [{name}] screenshot failed: {e}")
            continue

        for i, b in enumerate(blobs):
            fn = os.path.join(IMG_DIR, f"{name}_{tau:.2f}_v{i}.png")
            with open(fn, "wb") as fh:
                fh.write(b)

        rec = by_name.get(name)
        label = f"{name}   tau={tau:.2f}"
        if rec:
            label += (f"   E={rec['energy_total']:.2e}"
                      f"   dev={rec['contact_dev']:.2e}")
        grid_rows.append(blobs)
        grid_labels.append(label)
        print(f"  [{name}] captured {len(blobs)} views")

    if grid_rows:
        gpath = os.path.join(IMG_DIR, f"grid_tau{tau:.2f}.png")
        kv.save_grid(gpath, grid_rows, labels=grid_labels)
        print(f"  -> wrote {gpath}")

print(f"\nDone. Images in {IMG_DIR}")

[kayviz] Viewer already open — http://127.0.0.1:8002/?app=script
viewer connected — capturing

  tau = 0.5
  [bumpy_patch.out] screenshot failed: screenshot failed: no browser connected. Open the viewer tab (ws.show()) so a browser is connected.
  [coiled_torus_segment.out] screenshot failed: screenshot failed: no browser connected. Open the viewer tab (ws.show()) so a browser is connected.
  [flared_nautilus.out] screenshot failed: screenshot failed: no browser connected. Open the viewer tab (ws.show()) so a browser is connected.
  [helical_superellipsoid_tube.out] screenshot failed: screenshot failed: no browser connected. Open the viewer tab (ws.show()) so a browser is connected.
  [loft_00014.out] screenshot failed: screenshot failed: no browser connected. Open the viewer tab (ws.show()) so a browser is connected.
  [nautilus.out] screenshot failed: screenshot failed: no browser connected. Open the viewer tab (ws.show()) so a browser is connected.
  [ribbed_conch.out] screenshot fa

: 